In [2]:
import pandas as pd
import os

In [5]:
#Load Data
demographics = pd.read_csv("data/data_paired/csv/Demographics_Survey_filtered.csv")
train_split = pd.read_csv("data/data_paired/processed_audio/healthcode_10fold_train.csv")
valtest_split = pd.read_csv("data/data_paired/processed_audio/healthcode_10fold_val_test_paired_only.csv")

In [12]:
print ("demographics dataset shape:", demographics.shape)
demographics.head()

demographics dataset shape: (6668, 2)


,healthCode,professional-diagnosis
0,639e8a78-3631-4231-bda1-c911c1b169e5,0
1,52fe366a-2a9f-4260-9fb1-0fbc637a6cf4,0
2,67bdd316-26fc-4fc7-8431-bf9f41a649dd,0
3,45b4e2ca-8d15-4736-828c-829e3d4177f4,0
4,114fe75a-dc53-4b40-a987-0cddc76aff92,0


In [13]:
print ("train split shape:", train_split.shape)
train_split.head()

train split shape: (9840, 4)


,fold_iteration,healthCode,subset,modality_status
0,0,00ed2cf3-7685-4d55-bf9d-001fa233c09f,train,paired
1,0,016cd339-2d22-4fed-92bb-2341c9a46be0,train,paired
2,0,016d2897-5c3e-4a55-ad06-0677668946fa,train,paired
3,0,0177a257-8a43-4d56-aa12-8b3965690aac,train,paired
4,0,01aea090-d2cd-409e-8ce8-22887ada7af8,train,paired


In [14]:
print ("val/test split shape:", valtest_split.shape)
valtest_split.head()

val/test split shape: (3402, 4)


,fold_iteration,healthCode,subset,modality_status
0,0,00a78d55-a227-4944-b6e2-a8a8f9034f24,val,paired
1,0,0160664a-f4af-4071-a4aa-2967f3ea0503,val,paired
2,0,01b1ec31-0348-4148-a641-626d4391b4fb,val,paired
3,0,02baa3a2-fb40-4a86-94b4-0411125179f1,val,paired
4,0,02bb6e94-69c8-41c7-a85f-618624d7b5d5,val,paired


In [ ]:
#Compare healthCodes across datasets
demo_codes = set(demographics["healthCode"].unique())
train_codes = set(train_split["healthCode"].unique())
valtest_codes = set(valtest_split["healthCode"].unique())

print(f"Demographics_Survey: {len(demo_codes)} unique healthCodes")
print(f"Train split: {len(train_codes)} unique healthCodes")
print(f"Val/Test split: {len(valtest_codes)} unique healthCodes")

Demographics_Survey: 6668 unique healthCodes
Train split: 1701 unique healthCodes
Val/Test split: 1701 unique healthCodes


**Tapping Features Extraction**

In [3]:
tapping_sequences = pd.read_csv("data/tapping_event_sequences.csv")
tapping_features = pd.read_csv("data/tapping_statistical_features.csv")

print("Tapping event sequences shape:", tapping_sequences.shape)
print(tapping_sequences.head())

print("\nTapping statistical features shape:", tapping_features.shape)
print(tapping_features.head())

Tapping event sequences shape: (16596865, 3)
                             healthCode   delta_t             button
0  3e7e6085-a641-4b0f-bcda-c4fb3c0d3fe4  0.011159   TappedButtonNone
1  3e7e6085-a641-4b0f-bcda-c4fb3c0d3fe4  0.115568   TappedButtonNone
2  3e7e6085-a641-4b0f-bcda-c4fb3c0d3fe4  0.016707  TappedButtonRight
3  3e7e6085-a641-4b0f-bcda-c4fb3c0d3fe4  0.000610   TappedButtonLeft
4  3e7e6085-a641-4b0f-bcda-c4fb3c0d3fe4  0.117276   TappedButtonNone

Tapping statistical features shape: (55917, 7)
    mean_dt    std_dt  lr_ratio  left_count  right_count  total_taps  \
0  0.057085  0.047970  0.988636          87           88         351   
1  0.059368  0.056756  1.024096          85           83         337   
2  0.103956  0.075045  1.021277          48           47         192   
3  0.055758  0.032571  1.011236          90           89         359   
4  0.101238  0.047212  1.010204          99           98         197   

                             healthCode  
0  3e7e6085-a641-4

In [8]:
tapping_with_diagnosis = tapping_features.merge(
    demographics[["healthCode", "professional-diagnosis"]], 
    on="healthCode", 
    how="left"
)

print("Tapping features merged with diagnosis:")
print(tapping_with_diagnosis.head())

Tapping features merged with diagnosis:
    mean_dt    std_dt  lr_ratio  left_count  right_count  total_taps  \
0  0.057085  0.047970  0.988636          87           88         351   
1  0.059368  0.056756  1.024096          85           83         337   
2  0.103956  0.075045  1.021277          48           47         192   
3  0.055758  0.032571  1.011236          90           89         359   
4  0.101238  0.047212  1.010204          99           98         197   

                             healthCode  professional-diagnosis  
0  3e7e6085-a641-4b0f-bcda-c4fb3c0d3fe4                       1  
1  1ee9c522-d68a-4d32-8113-4c5246986978                       1  
2  8745e5f5-648b-4d46-bc7d-045fa6e2d628                       1  
3  5134777c-3563-41a5-abcf-cce272a2ebf3                       1  
4  faf76a8e-045c-427f-951d-8e1918f195a8                       0  


In [9]:
print("Mean Δt by diagnosis:")
print(tapping_with_diagnosis.groupby("professional-diagnosis")["mean_dt"].describe())

print("\nL/R ratio by diagnosis:")
print(tapping_with_diagnosis.groupby("professional-diagnosis")["lr_ratio"].describe())

Mean Δt by diagnosis:
                          count      mean       std       min       25%  \
professional-diagnosis                                                    
0                       16964.0  0.085263  0.057215  0.045963  0.052575   
1                       38953.0  0.088018  0.062543  0.045963  0.053298   

                             50%       75%       max  
professional-diagnosis                                
0                       0.057465  0.103295  0.306729  
1                       0.057465  0.103295  0.306729  

L/R ratio by diagnosis:
                          count      mean       std  min       25%       50%  \
professional-diagnosis                                                         
0                       16964.0  1.008213  0.055675  0.8  0.989011  1.010526   
1                       38953.0  1.004773  0.059995  0.8  0.987179  1.010204   

                             75%     max  
professional-diagnosis                    
0                       1

Count is the number of records (trials) per diagnosis class. 
- Diagnosis 0 (Healthy): 16,964 trial records
- Diagnosis 1 (PD): 38,953 trial records

50% is the median
For diagnosis 0 (Healthy): 
- 25% = 0.052575 → 25% of healthy trials have Δt ≤ 0.052575 seconds
- 50% = 0.057465 → 50% have Δt ≤ 0.057465 (this is the median)
- 75% = 0.103295 → 75% have Δt ≤ 0.103295 seconds